# 01 - Fixed-fleet smart-meter producer

Publishes exactly 10,000 detailed v2 events per completed 15-minute interval. OCI partial throttling is retried safely.

In [ ]:
import hashlib, json, math, random, re, time, uuid
from base64 import b64encode
from datetime import datetime, timedelta, timezone
from pathlib import Path
import oci

FLEET_SIZE, BATCH_SIZE, MAX_RETRIES = 10_000, 250, 12
NAMESPACE = uuid.UUID('7ea59664-2b40-4b4d-a739-29406c3c25ae')
config = {}
for raw in Path('/Workspace/ORACLE_AIDP_STREAMING/config/runtime.yaml').read_text(encoding='utf-8').splitlines():
    line = raw.strip()
    if line and not line.startswith('#') and ':' in line:
        key, value = line.split(':', 1); config[key.strip()] = value.strip().strip(chr(34)).strip(chr(39))
required = ('OCI_STREAM_ENDPOINT','OCI_STREAM_OCID','OCI_CONFIG_FILE','OCI_PROFILE')
missing = [key for key in required if not config.get(key)]
if missing: raise ValueError('runtime.yaml is missing: ' + ', '.join(missing))
if config.get('METER_COUNT') and int(config['METER_COUNT']) != FLEET_SIZE: raise ValueError('METER_COUNT must be 10000 or omitted.')

def completed_interval(value):
    now = datetime.fromisoformat(value.replace('Z','+00:00')) if value else datetime.now(timezone.utc) - timedelta(minutes=15)
    now = now.astimezone(timezone.utc).replace(second=0,microsecond=0)
    return now - timedelta(minutes=now.minute % 15)

def event(n, start):
    meter = f'MTR-{n:06d}'; rng = random.Random(int(hashlib.sha256(f'v2|{meter}|{start.isoformat()}'.encode()).hexdigest()[:16],16))
    segment = ('RESIDENTIAL','SMALL_BUSINESS','INDUSTRIAL')[n % 3]; temp = 28 + 7*math.sin((start.hour-13)*math.pi/12) + rng.uniform(-1.5,1.5)
    profile = {'RESIDENTIAL':.45,'SMALL_BUSINESS':1.2,'INDUSTRIAL':3.0}[segment]; kwh=round(max(.02,profile*(.35+.85*max(0,math.sin((start.hour-8)*math.pi/12)))*(.88 if start.weekday()>=5 else 1)+rng.gauss(0,profile*.035)),4)
    voltage=round(230+rng.gauss(0,2.5),2); pf=round(min(1,max(.78,.965+rng.gauss(0,.012))),3); demand=round(max(.05,kwh*4*rng.uniform(.94,1.12)),4)
    outage=15 if rng.random()<.0002 else 0; tamper=rng.random()<.0001
    return {'schema_version':'2.0','event_id':str(uuid.uuid5(NAMESPACE,f'{meter}|{start.isoformat()}')),'event_type':'INTERVAL_READING','event_ts_utc':(start+timedelta(minutes=15,seconds=rng.randint(1,90))).isoformat(),'meter_id':meter,'device_id':f'DEV-{n:06d}','service_point_id':f'SP-{n:06d}','service_point_type':'ELECTRIC','interval_start_utc':start.isoformat(),'interval_end_utc':(start+timedelta(minutes=15)).isoformat(),'interval_minutes':15,'consumption_kwh':kwh,'demand_kw':demand,'voltage_v':voltage,'current_a':round(demand*1000/voltage/pf,3),'power_factor':pf,'frequency_hz':round(50+rng.gauss(0,.03),3),'temperature_c':round(temp,2),'humidity_pct':round(min(95,max(20,63-1.2*(temp-28)+rng.gauss(0,4))),2),'quality_code':'ACTUAL','tariff_code':'PEAK' if 17<=start.hour<22 else 'SHOULDER' if 7<=start.hour<17 else 'OFF_PEAK','meter_status':'ACTIVE','firmware_version':f'2.{n%4+1}.0','region_code':f'REG-{(n-1)%20+1:02d}','feeder_id':f'FDR-{(n-1)%200+1:03d}','transformer_id':f'TRF-{(n-1)%1000+1:04d}','customer_segment':segment,'outage_minutes':outage,'tamper_flag':tamper,'measurement_events':[],'device_events':(['OUTAGE_RESTORED'] if outage else [])+(['TAMPER_ALERT'] if tamper else [])}

def wait_seconds(errors, attempt):
    waits = [int(x) for message in errors for x in re.findall(r'(\d+)\s*ms', message or '')]
    return max(.05, (max(waits, default=0)/1000.0), .1*(2**attempt))

def publish_with_retry(rows, batch_no):
    pending = rows
    for attempt in range(MAX_RETRIES):
        entries = [oci.streaming.models.PutMessagesDetailsEntry(key=b64encode(row['meter_id'].encode()).decode(),value=b64encode(json.dumps(row,separators=(',',':')).encode()).decode()) for row in pending]
        results = client.put_messages(config['OCI_STREAM_OCID'],oci.streaming.models.PutMessagesDetails(messages=entries)).data.entries
        failed = [(row,result.error_message or 'OCI rejected message') for row,result in zip(pending,results) if result.error]
        if not failed: return
        errors = [error for _,error in failed]
        if any('throttl' not in error.lower() for error in errors): raise RuntimeError(f'OCI rejected batch {batch_no}: {errors[:3]}')
        pending = [row for row,_ in failed]; delay = wait_seconds(errors,attempt)
        print(f'Batch {batch_no}: retrying {len(pending)} throttled events in {delay:.2f}s (attempt {attempt+1}/{MAX_RETRIES})')
        time.sleep(delay)
    raise RuntimeError(f'OCI continued throttling batch {batch_no} after {MAX_RETRIES} retries; rerun safely for the same interval.')

start = completed_interval(config.get('INTERVAL_START_UTC',''))
events = [event(n,start) for n in range(1,FLEET_SIZE+1)]
client = oci.streaming.StreamClient(oci.config.from_file(config['OCI_CONFIG_FILE'],config['OCI_PROFILE']),service_endpoint=config['OCI_STREAM_ENDPOINT'])
for offset in range(0,FLEET_SIZE,BATCH_SIZE): publish_with_retry(events[offset:offset+BATCH_SIZE],offset//BATCH_SIZE+1)
print(f'Published {FLEET_SIZE:,} unique meter events for {start.isoformat()} in {FLEET_SIZE//BATCH_SIZE} batches.')